# Colab A100 Training for Document-to-Markdown

Use this notebook after copying the repo and local `data/` directory to Google Drive. For speed, the notebook copies the project from Drive to Colab's local `/content` disk before training, then syncs weights/results back to Drive.

## 1. Select Runtime

In Colab: `Runtime -> Change runtime type -> GPU`, then select a premium GPU such as A100 if available.

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 2. Mount Drive and Copy Project to Local Disk

Edit `DRIVE_PROJECT` to the folder containing this repo and `data/train`, `data/val`, `data/test`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/docmd-reconstruction'  # TODO: adjust
LOCAL_PROJECT = '/content/docmd-reconstruction'

!rm -rf "$LOCAL_PROJECT"
!rsync -a --info=progress2 --exclude '.venv' --exclude '.uv-cache' "$DRIVE_PROJECT/" "$LOCAL_PROJECT/"
%cd "$LOCAL_PROJECT"

## 3. Install Project Dependencies

Colab usually provides CUDA PyTorch already. This uses `uv` for project dependencies and avoids any dataset download logic.

In [ ]:
!python -m pip install -q uv
!uv pip install --system -e .
!python -c "from ultralytics import YOLO; import torch; print(torch.__version__, torch.cuda.is_available())"

## 4. Ensure YOLO Datasets Exist

Run conversion only if the `data/yolo_*` folders were not already copied from your local machine.

In [ ]:
from pathlib import Path
if not Path('data/yolo_detection/dataset_detection.yaml').exists():
    !python scripts/convert_coco_to_yolo_detection.py
if not Path('data/yolo_segmentation/dataset_segmentation.yaml').exists():
    !python scripts/convert_coco_to_yolo_segmentation.py

## 5. TensorBoard

Start TensorBoard before or during training. Ultralytics writes `results.csv`; the bridge below mirrors it into TensorBoard event files.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/training

## 6. Full-Data Detection Training

Recommended A100 starting point: full data, 80 epochs, batch 16 at 1024px. If memory is comfortable, try batch 24. If Colab gives a smaller GPU, drop batch to 8.

In [ ]:
get_ipython().system_raw('python scripts/csv_to_tensorboard.py outputs/training/yolo26m_detection_full_a100 --interval 15 &')
!python scripts/train_yolo_detection.py \
  --model models/yolo26m.pt \
  --epochs 80 \
  --batch 16 \
  --imgsz 1024 \
  --device cuda \
  --workers 8 \
  --fraction 1.0 \
  --patience 25 \
  --project outputs/training \
  --name yolo26m_detection_full_a100 \
  --cos-lr \
  --save-period 5

## 7. Validate Detection

In [ ]:
!python scripts/evaluate.py \
  --yolo-weights outputs/training/yolo26m_detection_full_a100/weights/best.pt \
  --data data/yolo_detection/dataset_detection.yaml \
  --task detect \
  --device cuda \
  --imgsz 1024 \
  --output outputs/evaluation/yolo26m_detection_full_a100_val.json

## 8. Full-Data Segmentation Training

Segmentation is heavier. Start with batch 8 at 1024px on A100; drop to 4 on smaller GPUs.

In [ ]:
get_ipython().system_raw('python scripts/csv_to_tensorboard.py outputs/training/yolo26m_segmentation_full_a100 --interval 15 &')
!python scripts/train_yolo_segmentation.py \
  --model models/yolo26m-seg.pt \
  --epochs 60 \
  --batch 8 \
  --imgsz 1024 \
  --device cuda \
  --workers 8 \
  --fraction 1.0 \
  --patience 20 \
  --project outputs/training \
  --name yolo26m_segmentation_full_a100 \
  --cos-lr \
  --save-period 5

## 9. Validate Segmentation and Sync Results Back to Drive

In [ ]:
!python scripts/evaluate.py \
  --yolo-weights outputs/training/yolo26m_segmentation_full_a100/weights/best.pt \
  --data data/yolo_segmentation/dataset_segmentation.yaml \
  --task segment \
  --device cuda \
  --imgsz 1024 \
  --output outputs/evaluation/yolo26m_segmentation_full_a100_val.json

!mkdir -p "$DRIVE_PROJECT/outputs"
!rsync -a outputs/training "$DRIVE_PROJECT/outputs/"
!rsync -a outputs/evaluation "$DRIVE_PROJECT/outputs/"

## Resume After Interruption

If Colab disconnects, copy/sync outputs back from Drive, then resume from the last checkpoint:

```bash
python scripts/train_yolo_detection.py --model outputs/training/yolo26m_detection_full_a100/weights/last.pt --resume --device cuda
python scripts/train_yolo_segmentation.py --model outputs/training/yolo26m_segmentation_full_a100/weights/last.pt --resume --device cuda
```